# 03-dataframe-basics

03-dataframe-basics/01-select-filter.py
Demonstrates select, filter, withColumn, rename, drop, sort, dedup,
CASE WHEN, Spark SQL equivalents, and schema inspection.

Patterns covered:
  1.  select — string / col() / df.colname / expression
  2.  alias — rename columns inline
  3.  filter / where — single condition, compound, isin
  4.  withColumn — add / overwrite
  5.  withColumnRenamed
  6.  drop
  7.  orderBy / sort — desc, desc_nulls_last
  8.  distinct + dropDuplicates
  9.  limit
  10. printSchema + dtypes
  11. CASE WHEN via when/otherwise — surfaces known data flaws
  12. Spark SQL equivalent of CASE WHEN
  13. collect_list via groupBy+agg
  14. toDF — rename all columns at once

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views

from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = get_spark("03-dataframe-basics")
dfs   = register_views(spark)
emp   = dfs["employees"]
dept  = dfs["departments"]

## 1. select — string / col() / expression

In [ ]:
print("=" * 60)
print("1. select — string columns")
print("=" * 60)
emp.select("emp_id", "first_name", "last_name").show(5)

print("select — col() and arithmetic expression")
emp.select(F.col("emp_id"), (F.col("salary") * 1.1).alias("salary_plus_10pct")).show(5)

## 2. alias — rename columns inline

In [ ]:
print("=" * 60)
print("2. alias — rename and concatenate columns")
print("=" * 60)
emp.select(
    F.col("emp_id").alias("id"),
    F.concat(F.col("first_name"), F.lit(" "), F.col("last_name")).alias("full_name")
).show(5)

## 3. filter / where — single, compound, isin

In [ ]:
print("=" * 60)
print("3a. filter — single condition (dept_id == 1)")
print("=" * 60)
emp.filter(F.col("dept_id") == 1).show(5)

print("3b. where — SQL string predicate (salary > 150000)")
emp.where("salary > 150000").show(5)

print("3c. filter — compound AND condition (dept_id == 1 AND salary > 100000)")
emp.filter((F.col("dept_id") == 1) & (F.col("salary") > 100000)).show(5)

print("3d. filter — isin('Active', 'Terminated')")
emp.filter(F.col("status").isin("Active", "Terminated")).select(
    "emp_id", "first_name", "status", "termination_date"
).show(3)

## 4. withColumn — add / overwrite column

In [ ]:
print("=" * 60)
print("4a. withColumn — add annual_bonus (10% of salary)")
print("=" * 60)
emp.withColumn("annual_bonus", F.col("salary") * 0.10).select(
    "emp_id", "salary", "annual_bonus"
).show(5)

print("4b. withColumn — overwrite status as uppercase")
emp.withColumn("status_upper", F.upper(F.col("status"))).select(
    "emp_id", "status", "status_upper"
).show(3)

## 5. withColumnRenamed

In [ ]:
print("=" * 60)
print("5. withColumnRenamed — emp_id → employee_id")
print("=" * 60)
emp.withColumnRenamed("emp_id", "employee_id").printSchema()

## 6. drop

In [ ]:
print("=" * 60)
print("6. drop — remove phone and email columns")
print("=" * 60)
emp.drop("phone", "email").printSchema()

## 7. orderBy / sort

In [ ]:
print("=" * 60)
print("7a. orderBy — salary DESC (NULLs float to top by default)")
print("=" * 60)
emp.orderBy(F.col("salary").desc()).select("emp_id", "first_name", "salary").show(5)

print("7b. orderBy — salary DESC, NULLs last (emp 10 & 15 pushed to end)")
emp.orderBy(F.col("salary").desc_nulls_last()).select("emp_id", "salary").show(5)

## 8. distinct + dropDuplicates

In [ ]:
print("=" * 60)
print("8a. distinct — unique dept_id values")
print("=" * 60)
emp.select("dept_id").distinct().show()

print("8b. dropDuplicates — distinct (dept_id, status) combinations")
emp.select("dept_id", "status").dropDuplicates().orderBy("dept_id").show()

## 9. limit

In [ ]:
print("=" * 60)
print("9. limit — 5 earliest hires (orderBy hire_date)")
print("=" * 60)
emp.orderBy("hire_date").limit(5).select(
    "emp_id", "first_name", "last_name", "hire_date"
).show()

## 10. printSchema and dtypes

In [ ]:
print("=" * 60)
print("10. printSchema and dtypes")
print("=" * 60)
emp.printSchema()
print("dtypes list:")
for col_name, dtype in emp.dtypes:
    print(f"   {col_name:<20s}  {dtype}")

## 11. CASE WHEN via when/otherwise — surfaces data flaws

In [ ]:
print("=" * 60)
print("11. CASE WHEN — salary_band (surfaces NULL and Zero/Flaw rows)")
print("=" * 60)
print("""
   Expected findings:
     Unknown   → emp 10 (contractor, NULL salary) + emp 15 (new hire, NULL salary)
     Zero/Flaw → emp 19 (salary=0.0, soft-dup of emp 18)
""")
emp.withColumn(
    "salary_band",
    F.when(F.col("salary").isNull(), "Unknown")
     .when(F.col("salary") == 0, "Zero/Flaw")
     .when(F.col("salary") < 80000, "Junior")
     .when(F.col("salary") < 130000, "Mid")
     .when(F.col("salary") < 180000, "Senior")
     .otherwise("Lead")
).groupBy("salary_band").count().orderBy("count", ascending=False).show()

## 12. Spark SQL equivalent

In [ ]:
print("=" * 60)
print("12. Spark SQL equivalent of salary_band CASE WHEN")
print("=" * 60)
spark.sql("""
    SELECT
        CASE
            WHEN salary IS NULL THEN 'Unknown'
            WHEN salary = 0     THEN 'Zero/Flaw'
            WHEN salary < 80000  THEN 'Junior'
            WHEN salary < 130000 THEN 'Mid'
            WHEN salary < 180000 THEN 'Senior'
            ELSE 'Lead'
        END AS salary_band,
        COUNT(*) AS cnt
    FROM employees
    GROUP BY 1
    ORDER BY cnt DESC
""").show()

## 13. collect_list via groupBy + agg

In [ ]:
print("=" * 60)
print("13. collect_list — team last-names per dept_id")
print("=" * 60)
emp.groupBy("dept_id").agg(
    F.collect_list("last_name").alias("team")
).orderBy("dept_id").show(truncate=False)

## 14. toDF — rename all columns at once

In [ ]:
print("=" * 60)
print("14. toDF — rename all columns positionally")
print("=" * 60)
emp.select("emp_id", "first_name", "last_name").toDF("id", "fname", "lname").show(3)

stop_and_wait(spark)